# Tutorial 7.1: scSLAT Registration of COAD RNA and CODEX Data

This tutorial prepares human colon adenocarcinoma (COAD) data from the [SPATCH study](https://doi.org/10.1038/s41467-025-64292-3), which benchmarked high-throughput spatial transcriptomics platforms at subcellular resolution. Here, 5,001-gene Xenium RNA from 405,927 cells and 16-plex CODEX protein from 266,625 cells were acquired from adjacent serial sections; the processed data are publicly available through [SPATCH](http://spatch.pku-genomics.org/).

Because the modalities were measured on adjacent sections, their cells do not have direct location-matched RNA-protein pairs. [SLAT](https://doi.org/10.1038/s41467-023-43105-5) provides the heterogeneous spatial registration used here; its [official code](https://github.com/gao-lab/SLAT) and [documentation](https://slat.readthedocs.io/) describe the scSLAT implementation. High-confidence RNA-to-CODEX correspondences are projected to CODEX coordinates to create the real incomplete RNA-on-CODEX object used by Tutorial 7.2.


### GLUE preparation

scSLAT uses a shared molecular representation to register heterogeneous spatial modalities. The GLUE embeddings were generated before this workflow, so the tutorial starts from the prepared COAD AnnData objects.


In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
from scSLAT.model import Cal_Spatial_Net, load_anndatas, run_SLAT, spatial_match
from scSLAT.viz import match_3D_multi

plt.rcParams.update({"font.size": 11, "axes.linewidth": 1.0, "pdf.fonttype": 42})

### Load prepared inputs

The input objects retain their native section coordinates, precomputed `X_glue` embeddings and the original data-provided `spatial_cluster` annotations. The annotations are used for visual inspection only and do not enter the registration.


In [ ]:
# Read the prepared SLAT inputs
DATASET_DIR = Path("Datasets") / "COAD"
adata1 = sc.read_h5ad(DATASET_DIR / "adata.h5ad")
adata2 = sc.read_h5ad(DATASET_DIR / "adata_codex.h5ad")

In [ ]:
# Build spatial graphs
Cal_Spatial_Net(adata1, k_cutoff=30, model="KNN")
Cal_Spatial_Net(adata2, k_cutoff=20, model="KNN")

Section-specific K-nearest-neighbour graphs encode local spatial context before cross-section registration.

In [ ]:
# Run SLAT with the precomputed GLUE representation
edges, features = load_anndatas([adata1, adata2], feature="glue")
embd0, embd1, elapsed_time = run_SLAT(features, edges, LGCN_layer=6, hidden_size=4096)

### Match CODEX locations to RNA

Each CODEX location is assigned its best RNA candidate in the scSLAT embedding together with a cosine-similarity score. RNA is the reference section and CODEX is the query section.


In [ ]:
# Store SLAT embeddings and match CODEX protein to RNA
def to_numpy_embedding(embedding):
    return embedding.detach().cpu().numpy() if torch.is_tensor(embedding) else np.asarray(embedding)

adata1.obsm["X_slat"] = to_numpy_embedding(embd0).astype(np.float32, copy=False)
adata2.obsm["X_slat"] = to_numpy_embedding(embd1).astype(np.float32, copy=False)
# RNA is the reference and CODEX protein is the query.
best, index, distance = spatial_match([embd0, embd1], adatas=[adata1, adata2], reorder=False,
                                      smooth_range=200)
label_key = "spatial_cluster"
adata1_df = pd.DataFrame({"index": np.arange(embd0.shape[0]), "x": adata1.obsm["spatial"][:, 0],
                          "y": adata1.obsm["spatial"][:, 1], label_key: adata1.obs[label_key]})
adata2_df = pd.DataFrame({"index": np.arange(embd1.shape[0]), "x": adata2.obsm["spatial"][:, 0],
                          "y": adata2.obsm["spatial"][:, 1], label_key: adata2.obs[label_key]})
matching = np.array([np.arange(index.shape[0]), best])
best_match = distance[:, 0]

In [ ]:
# Select high-confidence RNA-CODEX registrations
MATCH_THRESHOLD = 0.85
fig, ax = plt.subplots(figsize=(5, 3.6))
n, bins, patches = ax.hist(distance[:, 0], bins=100, edgecolor="#8C9093", alpha=0.7)
for left, right, patch in zip(bins[:-1], bins[1:], patches):
    if 0.5 * (left + right) < MATCH_THRESHOLD:
        patch.set_facecolor("#8C9093")
        patch.set_edgecolor("#8C9093")
    else:
        patch.set_facecolor("#3990CE")
        patch.set_edgecolor("#3990CE")
ax.vlines(MATCH_THRESHOLD, 0, n.max(), color="#d62728", linestyles="dotted",
          label=f"Threshold = {MATCH_THRESHOLD:.2f}")
ax.set_xlabel("Cosine similarity")
ax.set_ylabel("CODEX protein cells")
ax.grid(False)
ax.legend(frameon=False)
plt.show()
matching_filter = matching[:, distance[:, 0] > MATCH_THRESHOLD]
print(f"High-confidence pairs: {matching_filter.shape[1]:,} / {adata2.n_obs:,}")

A similarity threshold retains supported RNA-CODEX correspondences. CODEX locations below the threshold are designated RNA-unregistered rather than treated as observed, mismatched RNA-protein pairs.

In [ ]:
# Inspect RNA-CODEX registration in 3D
adata1_vis = adata1_df[adata1_df[label_key].notna()].copy()
adata2_vis = adata2_df[adata2_df[label_key].notna()].copy()
valid_rna = set(adata1_vis["index"])
valid_codex = set(adata2_vis["index"])
valid_pairs = np.array([(codex in valid_codex) and (rna in valid_rna) for codex, rna in matching_filter.T])
matching_vis = matching_filter[:, valid_pairs]
multi_align = match_3D_multi(adata1_vis, adata2_vis, matching_vis, meta=label_key, rotate=["y", "y"],
                             scale_coordinate=True, subsample_size=300)
multi_align.draw_3D([5, 5], line_width=0.5, point_size=[0.0002, 0.0002], hide_axis=True, show_error=True)

### Save the unfiltered RNA reference

The unfiltered object transfers the highest-scoring RNA candidate to every CODEX coordinate. It provides a qualitative reference in Tutorial 7.2 and the complete starting object for the controlled simulation in Tutorial 7.3; it is not ground truth for real unregistered coordinates.


In [ ]:
# Save the unfiltered RNA-on-CODEX reference
# Transfer the highest-scoring RNA candidate to every CODEX location.
all_target_idx = matching[0].astype(int)
all_rna_idx = matching[1].astype(int)
if not np.array_equal(all_target_idx, np.arange(adata2.n_obs)):
    raise RuntimeError("Expected one best RNA match for every CODEX location.")
unfiltered_X = adata1[all_rna_idx, :].X
if sp.issparse(unfiltered_X):
    unfiltered_X = unfiltered_X.tocsr().astype(np.float32)
else:
    unfiltered_X = sp.csr_matrix(np.asarray(unfiltered_X, dtype=np.float32))
new_adata_matched = ad.AnnData(X=unfiltered_X, obs=adata2.obs.copy(), var=adata1.var.copy())
new_adata_matched.obs_names = adata2.obs_names.copy()
new_adata_matched.obsm["spatial"] = adata2.obsm["spatial"].copy()
new_adata_matched.obs["missing"] = "1"
new_adata_matched.obs["matched_RNA_index"] = all_rna_idx
new_adata_matched.obs["matched_RNA_barcode"] = adata1.obs_names.to_numpy()[all_rna_idx]
new_adata_matched.obs["SLAT_score"] = best_match[all_target_idx]
new_adata_matched.uns["registration"] = {"method": "scSLAT", "reference": "COAD RNA",
                                          "query": "COAD CODEX protein", "filtering": "none"}
new_adata_matched.write_h5ad(DATASET_DIR / "adata_matched.h5ad")

### Build registered RNA-on-CODEX data

The filtered object retains only high-confidence RNA-CODEX pairs. A CODEX location with `missing="1"` has a retained RNA registration, whereas `missing="0"` indicates an RNA-unregistered location; its zero RNA placeholder is not observed molecular data.


In [ ]:
# Create and save the CODEX-ordered registered RNA object
# Preserve CODEX coordinates; RNA-unregistered locations receive zero placeholders.
target_idx = matching_filter[0].astype(int)
align_idx = matching_filter[1].astype(int)
matched_expression = adata1[align_idx, :].X
if not sp.issparse(matched_expression):
    matched_expression = sp.csr_matrix(np.asarray(matched_expression, dtype=np.float32))
else:
    matched_expression = matched_expression.tocsr().astype(np.float32)
matched_expression = matched_expression.tocoo()
full_X = sp.csr_matrix((matched_expression.data,
    (target_idx[matched_expression.row], matched_expression.col)), shape=(adata2.n_obs, adata1.n_vars),
    dtype=np.float32)
full_obs = adata2.obs.copy()
full_obs["missing"] = "0"
full_obs["matched_RNA_barcode"] = "NA"
full_obs["SLAT_score"] = np.nan
full_obs.loc[adata2.obs_names[target_idx], "missing"] = "1"
full_obs.loc[adata2.obs_names[target_idx], "matched_RNA_barcode"] = adata1.obs_names.to_numpy()[align_idx]
full_obs.loc[adata2.obs_names[target_idx], "SLAT_score"] = best_match[target_idx]
new_adata_full = ad.AnnData(X=full_X, obs=full_obs, var=adata1.var.copy())
new_adata_full.obs_names = adata2.obs_names.copy()
new_adata_full.obsm["spatial"] = adata2.obsm["spatial"].copy()
new_adata_full.uns["registration"] = {"method": "scSLAT", "reference": "COAD RNA",
                                       "query": "COAD CODEX protein", "similarity_threshold": MATCH_THRESHOLD}
new_adata_full.write_h5ad(DATASET_DIR / "adata_reg.h5ad")

In [ ]:
# Display real registration missingness and transferred RNA expression
new_adata_full.uns["missing_colors"] = ["#d62728", "#1f77b4"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
sc.pl.embedding(new_adata_full, basis="spatial", color="missing", ax=axes[0], title="RNA registration status",
    size=2, legend_loc="right margin", show=False)
legend = axes[0].get_legend()
for text, label in zip(legend.get_texts(), ["RNA missing", "RNA registered"]):
    text.set_text(label)
for handle in getattr(legend, "legend_handles", getattr(legend, "legendHandles", [])):
    if hasattr(handle, "set_sizes"):
        handle.set_sizes([20.0])
sc.pl.embedding(new_adata_full, basis="spatial", color="ABAT", ax=axes[1],
    title="Representative gene: ABAT", size=2, cmap="viridis",
    mask_obs=new_adata_full.obs["missing"].astype(str) == "1", na_color="#B8B8B8", show=False)
for ax in axes:
    ax.set_xlabel("spatial1")
    ax.set_ylabel("spatial2")
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
plt.subplots_adjust(left=0.05, right=0.98, bottom=0.15, top=0.85, wspace=0.55)
plt.show()

The final panels show the spatial distribution of retained registrations and a representative transferred RNA feature.

### Registered and unfiltered outputs

`adata_matched.h5ad` contains the unfiltered RNA transfer at every CODEX coordinate. `adata_reg.h5ad` retains only high-confidence transfers and records RNA-unregistered CODEX locations as `missing="0"`; it is the real incomplete object used by Tutorial 7.2.
